## Anchor Texts for the Smart City Dimensions

In [1]:
import os
import json
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI

## Configure the API

In [ ]:
# Configure the API key (the key must never be written in the notebook)
# Option 1: environment variable OPENAI_API_KEY (or a .env file containing OPENAI_API_KEY=...)
# Option 2: if it does not exist, the key is requested when the cell is run
from getpass import getpass

load_dotenv()

API_KEY = os.environ.get("OPENAI_API_KEY") or getpass("OpenAI API key: ")
client = OpenAI(api_key=API_KEY)

print("OpenAI client configured.")

OpenAI client configured.


## Read the CSV file

In [3]:
ficheiro_base = "texto_base.csv"

df_base = pd.read_csv(
    ficheiro_base,
    sep=None,
    engine="python",
    encoding="latin1"
)

df_base

,dimensao,texto
0,Governança inteligente,Potenciar o acesso a serviços públicos digitai...
1,Sociedade inteligente,Promover a capacitação da sociedade a partir d...
2,Mobilidade Inteligente,Promover o investimento e o desenvolvimento de...
3,Ambiente Inteligente,Promover a criação de espaços públicos acessív...
4,Qualidade de vida inteligente,Implementar parcerias entre municípios e entid...
5,Economia inteligente,Potenciar a atratividade dos municípios para n...


In [4]:
# Clean the column names and texts
df_base.columns = df_base.columns.str.strip().str.lower()

df_base["dimensao"] = df_base["dimensao"].astype(str).str.strip()
df_base["texto"] = df_base["texto"].astype(str).str.replace("\r\n", " ", regex=False).str.strip()

df_base

,dimensao,texto
0,Governança inteligente,Potenciar o acesso a serviços públicos digitai...
1,Sociedade inteligente,Promover a capacitação da sociedade a partir d...
2,Mobilidade Inteligente,Promover o investimento e o desenvolvimento de...
3,Ambiente Inteligente,Promover a criação de espaços públicos acessív...
4,Qualidade de vida inteligente,Implementar parcerias entre municípios e entid...
5,Economia inteligente,Potenciar a atratividade dos municípios para n...


In [5]:
# Number of words in each text
df_base["num_palavras"] = df_base["texto"].str.split().apply(len)
df_base[["dimensao", "num_palavras"]]


,dimensao,num_palavras
0,Governança inteligente,106
1,Sociedade inteligente,163
2,Mobilidade Inteligente,147
3,Ambiente Inteligente,363
4,Qualidade de vida inteligente,312
5,Economia inteligente,148


## Normalise the dimension names

In [6]:
mapa_dimensoes = {
    "Governança inteligente": "Governança Inteligente",
    "Sociedade inteligente": "Sociedade Inteligente",
    "Mobilidade Inteligente": "Mobilidade Inteligente",
    "Ambiente Inteligente": "Ambiente Inteligente",
    "Qualidade de vida inteligente": "Qualidade de Vida Inteligente",
    "Economia inteligente": "Economia Inteligente"
}

df_base["dimensao"] = df_base["dimensao"].replace(mapa_dimensoes)

DIMENSOES = [
    "Governança Inteligente",
    "Sociedade Inteligente",
    "Mobilidade Inteligente",
    "Ambiente Inteligente",
    "Qualidade de Vida Inteligente",
    "Economia Inteligente"
]

df_base

,dimensao,texto,num_palavras
0,Governança Inteligente,Potenciar o acesso a serviços públicos digitai...,106
1,Sociedade Inteligente,Promover a capacitação da sociedade a partir d...,163
2,Mobilidade Inteligente,Promover o investimento e o desenvolvimento de...,147
3,Ambiente Inteligente,Promover a criação de espaços públicos acessív...,363
4,Qualidade de Vida Inteligente,Implementar parcerias entre municípios e entid...,312
5,Economia Inteligente,Potenciar a atratividade dos municípios para n...,148


### Definition of the anchor text versions

In [7]:
VERSOES_ANCORAS = {
    "v1_50_palavras": {
        "palavras_alvo": 50,
        "min_palavras": 45,
        "max_palavras": 60
    },
    "v2_100_palavras": {
        "palavras_alvo": 100,
        "min_palavras": 95,
        "max_palavras": 115
    },
    "v3_150_palavras": {
        "palavras_alvo": 150,
        "min_palavras": 140,
        "max_palavras": 170
    }
}

In [8]:
def gerar_texto_ancora_por_dimensao(
    dimensao,
    texto_base,
    palavras_alvo,
    min_palavras,
    max_palavras,
    modelo= "gpt-5.4-mini"
):
    """
    Generates an anchor text for a specific dimension,
    using as a base the text of that dimension in the CSV.
    """

    prompt = f"""
Tarefa:
Gerar um texto-âncora para embeddings semânticos.

Fonte:
Usa apenas o texto-base fornecido. Não acrescentes conhecimento externo.

Dimensão:
{dimensao}

Texto-base da dimensão:
\"\"\"
{texto_base}
\"\"\"

Regras:
- Usa apenas a informação presente no texto-base.
- Não acrescentes conhecimento externo.
- Escreve em português europeu.
- Produz um único parágrafo.
- O texto deve ser adequado para embeddings semânticos.
- O texto deve representar claramente a dimensão "{dimensao}".
- O texto deve ter aproximadamente {palavras_alvo} palavras.
- O texto deve ter entre {min_palavras} e {max_palavras} palavras.
- Evita listas, enumerações e bullets.
- Evita frases vagas como "esta dimensão é importante".
- Mantém linguagem clara e descritiva.
- Não devolvas explicações adicionais.

Devolve apenas JSON válido com esta estrutura:
{{
  "dimensao": "{dimensao}",
  "texto_ancora": "..."
}}
"""

    response = client.responses.create(
        model=modelo,
        input=prompt,
        max_output_tokens = int(palavras_alvo * 4) + 200

    )

    texto_resposta = response.output_text

    try:
        dados = json.loads(texto_resposta)
    except json.JSONDecodeError:
        print("The response was not valid JSON.")
        print("Response received:")
        print(texto_resposta)
        raise

    return dados

## Generate all the anchors


In [9]:
resultados = []

MODELO_LLM = "gpt-5.4-mini"

for nome_versao, config in VERSOES_ANCORAS.items():
    print(f"\nGenerating version: {nome_versao}")

    for _, row in df_base.iterrows():
        dimensao = row["dimensao"]
        texto = row["texto"]

        print(f"  - {dimensao}")

        resultado = gerar_texto_ancora_por_dimensao(
            dimensao=dimensao,
            texto_base=texto,
            palavras_alvo=config["palavras_alvo"],
            min_palavras=config["min_palavras"],
            max_palavras=config["max_palavras"],
            modelo=MODELO_LLM
        )

        resultado["versao"] = nome_versao
        resultados.append(resultado)

df_ancoras_versoes = pd.DataFrame(resultados)

df_ancoras_versoes


Generating version: v1_50_palavras
  - Governança Inteligente
  - Sociedade Inteligente
  - Mobilidade Inteligente
  - Ambiente Inteligente
  - Qualidade de Vida Inteligente
  - Economia Inteligente

Generating version: v2_100_palavras
  - Governança Inteligente
  - Sociedade Inteligente
  - Mobilidade Inteligente
  - Ambiente Inteligente
  - Qualidade de Vida Inteligente
  - Economia Inteligente

Generating version: v3_150_palavras
  - Governança Inteligente
  - Sociedade Inteligente
  - Mobilidade Inteligente
  - Ambiente Inteligente
  - Qualidade de Vida Inteligente
  - Economia Inteligente


,dimensao,texto_ancora,versao
0,Governança Inteligente,Potenciar o acesso a serviços públicos digitai...,v1_50_palavras
1,Sociedade Inteligente,A Sociedade Inteligente promove a capacitação ...,v1_50_palavras
2,Mobilidade Inteligente,Promover o investimento em infraestruturas e s...,v1_50_palavras
3,Ambiente Inteligente,"Promoção de espaços públicos acessíveis, inclu...",v1_50_palavras
4,Qualidade de Vida Inteligente,Qualidade de Vida Inteligente integra parceria...,v1_50_palavras
5,Economia Inteligente,A Economia Inteligente centra-se na atrativida...,v1_50_palavras
6,Governança Inteligente,A Governança Inteligente centra-se na potencia...,v2_100_palavras
7,Sociedade Inteligente,A Sociedade Inteligente assenta na capacitação...,v2_100_palavras
8,Mobilidade Inteligente,Promover o investimento e o desenvolvimento de...,v2_100_palavras
9,Ambiente Inteligente,"Promoção de espaços públicos acessíveis, inclu...",v2_100_palavras


## Validate the results

In [10]:
df_ancoras_versoes["num_palavras"] = df_ancoras_versoes["texto_ancora"].apply(
    lambda x: len(str(x).split())
)

# add the palavras_alvo column before displaying the DataFrame
df_ancoras_versoes["palavras_alvo"] = df_ancoras_versoes["versao"].map(
    {k: v["palavras_alvo"] for k, v in VERSOES_ANCORAS.items()}
)

df_ancoras_versoes[
    ["versao", "dimensao", "palavras_alvo", "num_palavras"]
]

,versao,dimensao,palavras_alvo,num_palavras
0,v1_50_palavras,Governança Inteligente,50,49
1,v1_50_palavras,Sociedade Inteligente,50,50
2,v1_50_palavras,Mobilidade Inteligente,50,52
3,v1_50_palavras,Ambiente Inteligente,50,48
4,v1_50_palavras,Qualidade de Vida Inteligente,50,46
5,v1_50_palavras,Economia Inteligente,50,48
6,v2_100_palavras,Governança Inteligente,100,100
7,v2_100_palavras,Sociedade Inteligente,100,106
8,v2_100_palavras,Mobilidade Inteligente,100,102
9,v2_100_palavras,Ambiente Inteligente,100,104


## Sort and save to CSV

In [11]:
df_validacao = df_ancoras_versoes.copy()

df_validacao["min_palavras"] = df_validacao["versao"].map(
    {k: v["min_palavras"] for k, v in VERSOES_ANCORAS.items()}
)

df_validacao["max_palavras"] = df_validacao["versao"].map(
    {k: v["max_palavras"] for k, v in VERSOES_ANCORAS.items()}
)

df_validacao["dentro_intervalo"] = (
    (df_validacao["num_palavras"] >= df_validacao["min_palavras"]) &
    (df_validacao["num_palavras"] <= df_validacao["max_palavras"])
)

df_validacao[
    ["versao", "dimensao", "palavras_alvo", "num_palavras", "dentro_intervalo"]
]

,versao,dimensao,palavras_alvo,num_palavras,dentro_intervalo
0,v1_50_palavras,Governança Inteligente,50,49,True
1,v1_50_palavras,Sociedade Inteligente,50,50,True
2,v1_50_palavras,Mobilidade Inteligente,50,52,True
3,v1_50_palavras,Ambiente Inteligente,50,48,True
4,v1_50_palavras,Qualidade de Vida Inteligente,50,46,True
5,v1_50_palavras,Economia Inteligente,50,48,True
6,v2_100_palavras,Governança Inteligente,100,100,True
7,v2_100_palavras,Sociedade Inteligente,100,106,True
8,v2_100_palavras,Mobilidade Inteligente,100,102,True
9,v2_100_palavras,Ambiente Inteligente,100,104,True


In [12]:
df_ancoras_versoes.to_csv(
    "ancoras_llm_3_versoes.csv",
    sep=";",
    index=False,
    encoding="utf-8-sig"
)

print("File saved: ancoras_llm_3_versoes.csv")

File saved: ancoras_llm_3_versoes.csv


In [13]:
for nome_versao in VERSOES_ANCORAS.keys():
    df_temp = df_ancoras_versoes[
        df_ancoras_versoes["versao"] == nome_versao
    ].copy()

    nome_ficheiro = f"ancoras_{nome_versao}.csv"

    df_temp.to_csv(
        nome_ficheiro,
        sep=";",
        index=False,
        encoding="utf-8-sig"
    )

    print(f"Saved: {nome_ficheiro}")

Saved: ancoras_v1_50_palavras.csv
Saved: ancoras_v2_100_palavras.csv
Saved: ancoras_v3_150_palavras.csv


* texto_base.csv
    <br>↓<br>
* read columns dimensao + texto
        ↓
* generate anchor text per dimension
        ↓
* save ancoras_v1_llm.csv
        ↓
* compute embeddings of the anchors
        ↓
* compare with title + description of the datasets

In [14]:
pd.set_option("display.max_colwidth", None)

df_ancoras_versoes[
    ["versao", "dimensao", "num_palavras", "texto_ancora"]
]

,versao,dimensao,num_palavras,texto_ancora
0,v1_50_palavras,Governança Inteligente,49,"Potenciar o acesso a serviços públicos digitais, portais e aplicações integradas, com recurso preferencial às arquiteturas de referência da Administração Pública, promovendo governação aberta através do portal participa.gov e do envolvimento do cidadão em living labs. Inclui serviços online, IoT, operações integradas, dados abertos e monitorização em tempo real."
1,v1_50_palavras,Sociedade Inteligente,50,"A Sociedade Inteligente promove a capacitação da sociedade através do ensino, da inclusão e da literacia digital, bem como de atividades de saúde, inclusão social, requalificação de pessoas socialmente excluídas e apoio social. Recorre a plataformas digitais para apelar ao voto e integra indicadores de inclusão, participação cívica e educação."
2,v1_50_palavras,Mobilidade Inteligente,52,"Promover o investimento em infraestruturas e sistemas inteligentes para impulsionar mobilidade inteligente e sustentável, com combustíveis de baixo teor de carbono e mobilidade elétrica. Fomentar transportes públicos mais sustentáveis e eficientes, com carregamento de veículos elétricos, mobilidade multimodal, sistemas interconectados e informação em tempo real para gestão do tráfego e dos semáforos."
3,v1_50_palavras,Ambiente Inteligente,48,"Promoção de espaços públicos acessíveis, inclusivos e verdes, da sustentabilidade energética e ambiental do edificado, da economia circular e das fontes renováveis, com gestão eficiente da água, resíduos, iluminação pública e monitorização integrada da qualidade do ar, clima, ruído e emissões, apoiando edifícios inteligentes e planeamento urbano sustentável."
4,v1_50_palavras,Qualidade de Vida Inteligente,46,"Qualidade de Vida Inteligente integra parcerias entre municípios e entidades de saúde, sistemas de proteção, segurança e emergências, habitação acessível e intergeracional, cultura, arte e desporto no planeamento urbano, comunidades de energia e soluções de proximidade como telessaúde e teleassistência, alinhadas com planos e diretrizes nacionais."
5,v1_50_palavras,Economia Inteligente,48,"A Economia Inteligente centra-se na atratividade dos municípios para novos residentes, famílias e setor empresarial, promovendo a residência e o investimento nos territórios. Apoia a economia local, o empreendedorismo, a inovação, a investigação e desenvolvimento, o comércio digital e a promoção da região como destino turístico de eleição."
6,v2_100_palavras,Governança Inteligente,100,"A Governança Inteligente centra-se na potenciação do acesso a serviços públicos digitais, portais e aplicações integradas, com recurso preferencial às arquiteturas de referência da Administração Pública. Inclui o reforço de iniciativas de governação aberta, através da promoção e alavancagem do portal participa.gov e do incentivo ao envolvimento do cidadão em living labs. Abrange também serviços online, como procedimentos online e disponibilidade de aplicações móveis às pessoas, a utilização de IoT em municípios com dispositivos ou sistemas interconectados monitorizados ou controlados remotamente, operações integradas com monitorização em tempo real e governo aberto através do fornecimento e da utilização de dados abertos."
7,v2_100_palavras,Sociedade Inteligente,106,"A Sociedade Inteligente assenta na capacitação da sociedade através do sistema de ensino e de iniciativas de inclusão e literacia digital, promovendo competências digitais e o acesso à educação superior. Inclui também ações de promoção da saúde e de inclusão social, orientadas para estilos de vida saudáveis, prevenção de comportamentos de risco, requalificação de pessoas socialmente excluídas e reforço da rede de apoio social. Integra o uso de plataformas digitais para apelar ao voto dos cidadãos, bem como indicadores de inclusão, participação cívica e educação, como a ligação à internet, a existência de plataformas eletrónicas de participação cívica e a taxa de votantes nas eleições 